# 02 · Level-1 merged analyzers: one full-year table

Merges the two analyzers into a single continuous full-year table of **Level-1
EddyPro fluxes** and time lags, before the flux processing chain (no L2 quality
flags, no storage term, no outlier removal, no USTAR filtering). This notebook
does the merging only and saves the result; the figures are drawn downstream in
[notebook 03](03_level1_merged_figures.ipynb), which reads nothing but this table.

The two analyzers cover complementary halves of 2021: the QCL runs in campaign
2021_1 (January to 20 July) and the LGR in campaign 2021_2 (22 July to December).
Concatenating them over time gives one continuous **full-year** series per variant.

Inputs:
- `data/01-eddypro_fluxes_level-1_parquet/` for the raw flux and the time lag
  (`*-1` to `*-4`).
- `data/01-pwb_tlag_summary_parquet/` for the PWB$_{OPT}$ lag (`*-5`).

Output:
- `data/02-level1_merged_parquet/level1_merged.parquet`: one table on the
  half-hourly timestamp index, with a raw flux and a time-lag column per variant
  and gas, plus an `ANALYZER` column naming the instrument each record comes from.

## Imports

In [1]:
from datetime import datetime
from pathlib import Path

import pandas as pd

from diive.core.io.files import load_parquet, save_parquet

NB_START = datetime.now()  # notebook start time (reported in the last cell)

## Configuration

Each variant is the QCL half and the LGR half of that variant merged into one
full-year series. The lag for variants `*-1` to `*-4` is the EddyPro time lag
used, while the PWB$_{OPT}$ variant (`*-5`) uses its optimised lag (the PWB
estimate after the S1/S2/S3 decision rule, column `*_tlag_final_pf_s` in the PWB
summaries). Both end up in the same `*_TLAG_USED_V*` column of the output table,
one column per variant, so downstream code reads the lag the same way for every
variant.

Only the data-side settings live here. Everything to do with how the merged series
are drawn (colors, variant labels, histogram raster, axis ranges) belongs to
[notebook 03](03_level1_merged_figures.ipynb).

In [2]:
LEVEL1DIR = Path("../data/01-eddypro_fluxes_level-1_parquet")  # raw flux + lag, *-1..*-4
PWBDIR = Path("../data/01-pwb_tlag_summary_parquet")           # lag, *-5
OUTDIR = Path("../data/02-level1_merged_parquet")              # merged full-year table
OUTNAME = "level1_merged"                                      # .parquet is added on save
OUTDIR.mkdir(parents=True, exist_ok=True)

YEAR = 2021

# The two analyzers cover complementary halves of the year; QCL is the earlier
# campaign, LGR the later one.
ANALYZERS = ["QCL", "LGR"]
# The five time-lag variants; each is merged across both analyzers.
VARIANTS = [1, 2, 3, 4, 5]
PWB_VARIANT = 5  # the PWB variant uses its detected lag, not the EddyPro lag

# Per gas: the flux variable, the EddyPro time-lag-used variable, and the PWB_OPT
# optimised-lag column.
GASES = {
    "N2O": {"flux": "FN2O", "lag": "N2O_TLAG_USED", "pwb_lag": "n2o_tlag_final_pf_s"},
    "CH4": {"flux": "FCH4", "lag": "CH4_TLAG_USED", "pwb_lag": "ch4_tlag_final_pf_s"},
}


def flux_col(gas, var):
    """Merged flux column name, e.g. ('N2O', 3) -> 'FN2O_V3'."""
    return f"{GASES[gas]['flux']}_V{var}"


def lag_col(gas, var):
    """Merged time-lag column name, e.g. ('N2O', 3) -> 'N2O_TLAG_USED_V3'. For the
    PWB variant the values come from the PWB summary, not from EddyPro."""
    return f"{GASES[gas]['lag']}_V{var}"


print(f"Merging {ANALYZERS} into one full-year table (Level-1 raw fluxes, {YEAR})")

Merging ['QCL', 'LGR'] into one full-year table (Level-1 raw fluxes, 2021)


## Load and merge the raw fluxes and the time lag

For each analyzer, variant and gas the raw Level-1 flux and the time lag are read
from the same EddyPro table (`*-1` to `*-4`); the PWB$_{OPT}$ lag (`*-5`) comes
from the PWB summaries. The time lag is then **masked to the records where the raw
flux is present**, so any later lag plot, histogram or mode describes exactly the
records that carry a flux. The two analyzers are then concatenated over time per
variant. Because the campaigns do not overlap, this is a clean stitch: no
timestamp appears in both halves, which is also what lets a single `ANALYZER`
column say where each record comes from.

In [3]:
def _yr(df):
    """Restrict a table to the analysis year."""
    return df.loc[df.index.year == YEAR]


columns = {}         # output column name -> merged full-year series
campaign_index = {}  # analyzer -> its timestamps (used for the ANALYZER column)

for var in VARIANTS:
    flux_parts = {gas: [] for gas in GASES}
    lag_parts = {gas: [] for gas in GASES}

    for analyzer in ANALYZERS:
        code = f"{analyzer}-{var}"
        # Level-1 EddyPro table: carries both the raw fluxes and the EddyPro lag.
        l1 = _yr(load_parquet(filepath=str(LEVEL1DIR / f"{code}.parquet")))
        prev = campaign_index.get(analyzer)
        campaign_index[analyzer] = l1.index if prev is None else prev.union(l1.index)
        # Lag source: PWB summary for the PWB variant, the Level-1 table otherwise.
        if var == PWB_VARIANT:
            lagdf = _yr(load_parquet(filepath=str(PWBDIR / f"{code}_pwb_tlag.parquet")))
        else:
            lagdf = l1
        for gas, v in GASES.items():
            fluxseries = l1[v["flux"]]
            lagcol = v["pwb_lag"] if var == PWB_VARIANT else v["lag"]
            lag = lagdf[lagcol].reindex(fluxseries.index)
            flux_parts[gas].append(fluxseries)
            # Keep the time lag only where the raw flux is present, so the lag
            # describes exactly the records that carry a flux.
            lag_parts[gas].append(lag.where(fluxseries.notna()))

    # Stitch the two campaigns into one full-year series per variant and gas.
    for gas in GASES:
        columns[flux_col(gas, var)] = pd.concat(flux_parts[gas]).sort_index()
        columns[lag_col(gas, var)] = pd.concat(lag_parts[gas]).sort_index()

merged = pd.DataFrame(columns).sort_index()
merged.index.name = "TIMESTAMP_MIDDLE"  # diive's load_parquet expects a named timestamp

# Which instrument each record comes from. The campaigns do not overlap, so this is
# unambiguous; it is what lets notebook 03 split the merged series by instrument
# and place the campaign handover.
analyzer_col = pd.Series(pd.NA, index=merged.index, dtype="object")
for analyzer in ANALYZERS:
    analyzer_col.loc[merged.index.isin(campaign_index[analyzer])] = analyzer
merged.insert(0, "ANALYZER", analyzer_col)

for analyzer in ANALYZERS:
    span = merged.index[merged["ANALYZER"] == analyzer]
    print(f"{analyzer}: {len(span)} records "
          f"({span.min():%Y-%m-%d %H:%M} to {span.max():%Y-%m-%d %H:%M})")
for var in VARIANTS:
    s = merged[flux_col("N2O", var)]
    print(f"variant -{var}: {int(s.notna().sum())} valid merged raw N2O")

> Loaded .parquet file ..\data\01-eddypro_fluxes_level-1_parquet\QCL-1.parquet (0.080 seconds).

> Loaded .parquet file ..\data\01-eddypro_fluxes_level-1_parquet\LGR-1.parquet (0.038 seconds).

> Loaded .parquet file ..\data\01-eddypro_fluxes_level-1_parquet\QCL-2.parquet (0.044 seconds).

> Loaded .parquet file ..\data\01-eddypro_fluxes_level-1_parquet\LGR-2.parquet (0.038 seconds).

> Loaded .parquet file ..\data\01-eddypro_fluxes_level-1_parquet\QCL-3.parquet (0.043 seconds).

...

> Loaded .parquet file ..\data\01-eddypro_fluxes_level-1_parquet\QCL-5.parquet (0.043 seconds).

> Loaded .parquet file ..\data\01-pwb_tlag_summary_parquet\QCL-5_pwb_tlag.parquet (0.008 seconds).

> Loaded .parquet file ..\data\01-eddypro_fluxes_level-1_parquet\LGR-5.parquet (0.038 seconds).

> Loaded .parquet file ..\data\01-pwb_tlag_summary_parquet\LGR-5_pwb_tlag.parquet (0.007 seconds).

QCL: 9632 records (2021-01-01 00:15 to 2021-07-20 15:45)
LGR: 7803 records (2021-07-22 10:45 to 2021-12-31 23:45)
variant -1: 17181 valid merged raw N2O
variant -2: 17181 valid merged raw N2O
variant -3: 17181 valid merged raw N2O
variant -4: 17181 valid merged raw N2O
variant -5: 17182 valid merged raw N2O


## Save the merged table

One file for the whole stage: `data/02-level1_merged_parquet/level1_merged.parquet`.
Columns are `ANALYZER`, then per variant and gas the merged raw flux
(`FN2O_V1` … `FCH4_V5`) and the merged time lag used (`N2O_TLAG_USED_V1` …
`CH4_TLAG_USED_V5`). Everything downstream of this notebook reads that one file
instead of the ten Level-1 tables.

**Table.** Summary statistics of the merged Level-1 table, one column per
variant and gas: the merged raw flux and the merged time lag used.

In [4]:
filepath = save_parquet(filename=OUTNAME, data=merged, outpath=str(OUTDIR))

print(f"\n{merged.shape[0]} rows x {merged.shape[1]} columns "
      f"({merged.index.min():%Y-%m-%d %H:%M} to {merged.index.max():%Y-%m-%d %H:%M})")
print(f"columns: {list(merged.columns)}")
merged.describe()

> Saved file ..\data\02-level1_merged_parquet\level1_merged.parquet (0.033 seconds).


17435 rows x 21 columns (2021-01-01 00:15 to 2021-12-31 23:45)
columns: ['ANALYZER', 'FN2O_V1', 'N2O_TLAG_USED_V1', 'FCH4_V1', 'CH4_TLAG_USED_V1', 'FN2O_V2', 'N2O_TLAG_USED_V2', 'FCH4_V2', 'CH4_TLAG_USED_V2', 'FN2O_V3', 'N2O_TLAG_USED_V3', 'FCH4_V3', 'CH4_TLAG_USED_V3', 'FN2O_V4', 'N2O_TLAG_USED_V4', 'FCH4_V4', 'CH4_TLAG_USED_V4', 'FN2O_V5', 'N2O_TLAG_USED_V5', 'FCH4_V5', 'CH4_TLAG_USED_V5']


,FN2O_V1,N2O_TLAG_USED_V1,FCH4_V1,CH4_TLAG_USED_V1,FN2O_V2,N2O_TLAG_USED_V2,FCH4_V2,...,N2O_TLAG_USED_V4,FCH4_V4,CH4_TLAG_USED_V4,FN2O_V5,N2O_TLAG_USED_V5,FCH4_V5,CH4_TLAG_USED_V5
count,17181.000000,17181.000000,17181.000000,17181.000000,17181.000000,17181.000000,17181.000000,...,17181.000000,17181.000000,17181.000000,17182.000000,17182.000000,17182.000000,17182.00000
mean,1.041457,3.529300,10.238387,4.434588,1.031666,3.206484,10.524306,...,1.117336,11.042825,1.144843,0.996013,1.250745,11.016649,1.25353
std,6.361722,3.404119,163.119960,3.788879,6.339842,3.110467,160.550262,...,0.572118,154.421006,0.547243,4.900372,0.600504,153.977769,0.60473
min,-374.774000,-0.050000,-7207.160000,-0.050000,-376.184000,0.050000,-7194.800000,...,0.600000,-6743.700000,0.650000,-330.058000,0.100000,-6718.930000,0.25000
25%,-0.078469,0.750000,-8.283100,0.750000,-0.068905,0.700000,-7.457690,...,0.600000,-4.862130,0.650000,0.004832,0.700000,-4.860392,0.70000
50%,0.294426,1.800000,3.071570,3.150000,0.284953,1.750000,2.872080,...,0.600000,2.265190,0.650000,0.220531,0.900000,2.259545,0.85000
75%,1.067310,6.550000,19.342500,8.550000,1.042320,5.400000,18.612400,...,1.750000,17.122000,1.750000,0.874446,1.800000,17.066900,1.85000
max,205.036000,10.000000,5272.880000,10.000000,205.498000,9.950000,5268.210000,...,1.750000,5245.000000,1.750000,169.874000,2.850000,5235.070000,2.60000


## Runtime

In [5]:
NB_END = datetime.now()
print(f"Start:    {NB_START:%Y-%m-%d %H:%M:%S}")
print(f"End:      {NB_END:%Y-%m-%d %H:%M:%S}")
print(f"Runtime:  {NB_END - NB_START}")

Start:    2026-07-23 16:23:18
End:      2026-07-23 16:23:19
Runtime:  0:00:01.158752
